# Calculating the EKE along the contour - Panan $0.05^o$

A previous analysis of the CSHT between simulations of different horizontal resolutions show that increasing the resolution do not increase the Cross-slope heat transport. However, the spatial pattern of the heat transport changes, with some regions importing more heat towards the shelf, and others exporting more heat. Perhaps as we change resolution, the CSHT from eddies increase in some regions and decrease in others, explaining the displacement. We willt ry to cross reference the CSHT with the EKE along the isobath. For that, we will need to calculate the EKE along the contour



this code calcualtes the EKE along the contour and save the values into netcdfs

In [1]:
import cosima_cookbook as cc
import matplotlib.pyplot as plt
import cmocean as cm
import numpy as np
from dask.distributed import Client
from scipy.interpolate import interp1d

import xarray as xr
import cf_xarray as cfxr

In [2]:
client = Client(memory_limit = '1400gb',n_workers = 24)

In [3]:
session = cc.database.create_session('/home/156/wf4500/databases/access/panan005_rerun.db') #panan005

imm= 12
monthdays = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
month = str(int(imm))
month = month.zfill(2)
year = str(2001)
start_time=year+'-'+month  

start_time0=year+'-'+month +'-01'     
end_time0=year+'-'+month +'-' + str(monthdays[int(int(imm)-1)])
print(start_time0) 
print(end_time0) 
#exp = 'panan01_yr2000_HTsigma' #panan01
exp = 'panan_005deg_jra55_ryf_2024_12_14'

print("Start date =" + start_time) 
year2=str(int(start_time[0:4])+1)
month2=str(int(start_time[5:7])+1)
month2=str(int(month2))
month2 = month2.zfill(2)    
print("month2 is =" + month2) 
print("year2 is =" + str(year2))     

imon = int(imm)
if imon <12:
	end_time=year+'-'+month2
else:
	end_time=year2+'-01'

print("End date =" + end_time) 


time_period = str(int(start_time[:4]))+'-'+str(int(end_time[:4]))

# reference density value:
rho_0 = 1035.0
# specific heat capacity of sea water:
cp = 3992.1
lat_range = slice(-90,-59)

isobath_depth = 1000

# pick a freezing point temp:
temp_freezing = -1.9

2001-12-01
2001-12-31
Start date =2001-12
month2 is =13
year2 is =2002
End date =2002-01


In [4]:
#Getting mass transport in kg/s
vmo_nt = cc.querying.getvar(exp,'vmo',session,ncfile='%month_rho%',start_time=start_time,end_time=end_time,chunks={}).sel(time=slice(start_time,end_time))
umo_et = cc.querying.getvar(exp,'umo',session,ncfile='%month_rho%',start_time=start_time,end_time=end_time,chunks={}).sel(time=slice(start_time,end_time))

vmo_nt = vmo_nt.sel(yq=lat_range).sel(time=slice(start_time0,end_time0))
umo_et = umo_et.sel(yh=lat_range).sel(time=slice(start_time0,end_time0))
vmo_nttime=vmo_nt.time


#importing volume
vol_centre = cc.querying.getvar(exp,'volcello',session,ncfile='%month_rho2.nc',start_time=start_time,end_time=end_time,chunks={})\
.sel(time=slice(start_time,end_time)).sel(yh=lat_range).sel(time=slice(start_time0,end_time0))

In [5]:
#Making volume on xq and yq grid  ######

#First making a halo for proper edges interpolation
lon_interval = vol_centre.xh.diff('xh').mean()
lon_rightappend = vol_centre.xh[-1].values+np.cumsum([lon_interval,lon_interval,lon_interval,lon_interval,lon_interval])
lon_leftappend = vol_centre.xh[0].values - \
np.flip(np.cumsum([lon_interval,lon_interval,lon_interval,lon_interval]))

newlon = np.concatenate((lon_leftappend,vol_centre.xh.values,lon_rightappend), axis=0)
vol_centre_halo = xr.concat([vol_centre.isel(xh=slice(-5,-1)),vol_centre,vol_centre.isel(xh=slice(0,5))], dim='xh')
vol_centre_halo['xh'] = newlon


#Now interpolating volumes onto the xq and yq grids
vol_yhxq = vol_centre_halo.interp(xh=umo_et.xq)
vol_yqxq = vol_centre_halo.interp(xh=umo_et.xq,yh=vmo_nt.yq)
vol_yqxh = vol_centre_halo.interp(yh=vmo_nt.yq)
vol_yhxh = vol_centre.copy()

importing dxt to calculate u velocity, and dyt to calculate v velocity and defining the boussinesq velocity

In [6]:
#dxs in all grid points
dx_xhyh = cc.querying.getvar(exp,'dxt',session,ncfile='19990101.ocean_static.nc')
dx_xqyh = cc.querying.getvar(exp,'dxCu',session,ncfile='19990101.ocean_static.nc')
dx_xhyq = cc.querying.getvar(exp,'dxCv',session,ncfile='19990101.ocean_static.nc')
dx_xqyq = dx_xhyh.rename({'xh':'xq','yh':'yq'}).copy()
dx_xqyq=  xr.concat([dx_xqyq[:,-1],dx_xqyq], dim='xq')
dx_xqyq['yq'] = dx_xhyq.yq[:-1]
dx_xqyq['xq'] = dx_xqyh.xq

#dys in all grid points
dy_xhyh = cc.querying.getvar(exp,'dyt',session,ncfile='19990101.ocean_static.nc')
dy_xqyh = cc.querying.getvar(exp,'dyCu',session,ncfile='19990101.ocean_static.nc')
dy_xhyq = cc.querying.getvar(exp,'dyCv',session,ncfile='19990101.ocean_static.nc')
dy_xqyq = dy_xhyh.rename({'xh':'xq','yh':'yq'}).copy()
dy_xqyq=  xr.concat([dy_xqyq[:,-1],dy_xqyq], dim='xq')
dy_xqyq['yq'] = dy_xhyq.yq[:-1]
dy_xqyq['xq'] = dy_xqyh.xq

#setting the reference rho|
rho_ref = 1035 #kg m^{-3}, used as standard by Pan-Antarctic

We will calculate $U$ speeds by:

$$
  U(xq,yh,\sigma)\ =\ {umo} \times \rho^{-1} \times {Vol}^{-1}_{(xq,yh)} \times dxt_{(xq,yh)} \ \ \ [m\ s^{-1}]
$$

and $V$ speeds by:

$$
  V(xh,yq,\sigma)\ =\ {vmo} \times \rho^{-1} \times {Vol}^{-1}_{(xh,yq)} \times dyt_{(xh,yq)} \ \ \ [m\ s^{-1}]
$$

In [7]:
#first for U
U_xq = (umo_et*(1/rho_ref) * (1/vol_yhxq) * dx_xqyh).compute()

#now for V
V_yq = (vmo_nt*(1/rho_ref) * (1/vol_yqxh) * dy_xhyq).compute()

/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/dask/core.py:127: RuntimeWarning: divide by zero encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/dask/core.py:127: RuntimeWarning: divide by zero encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/dask/core.py:127: RuntimeWarning: divide by zero encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/dask/core.py:127: RuntimeWarning: divide by zero encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/lib/python3.10/site-packages/dask/core.py:127: RuntimeWarning: divide by zero encountered in div

Now we can just extract the values along the isobath by interpolation. Not ideal, but that is the omnly way we can have bothU and V in all points along the contour

In [8]:
#importing heat transports for the grid along the isobath
#panan01 daily as f(z,time)
src_p01 = '/g/data/ik11/users/wf4500/Project_panan/GH/Panan_HT_ASC/Processed_data/panan01_rerun/CSHT_month_rhoonline/*'
#panan005 daily as f(z,time)
src_p005 = '/g/data/ik11/users/wf4500/Project_panan/GH/Panan_HT_ASC/Processed_data/panan005_rerun/CSHT_month_rhoonline/*'
#panan0025 daily as f(z,time)
src_p0025 = '/g/data/ik11/users/wf4500/Project_panan/GH/Panan_HT_ASC/Processed_data/panan0025/CSHT_month_rhoonline/*'

lat_slice  = slice(-83,-59)

figdir='/home/156/wf4500/v45_wf4500/Project_panan/GH/Panan_HT_ASC/figs/'


#importing panan01
import glob
gl =glob.glob(src_p01)
gl=sorted(gl)
print("%i netcdf data files"%len(gl))
p01_HTmean=xr.open_mfdataset(gl, concat_dim='time', combine='nested').mean('time')


#importing panan005
gl =glob.glob(src_p005)
gl=sorted(gl)
print("%i netcdf data files"%len(gl))
p005_HTmean=xr.open_mfdataset(gl, concat_dim='time', combine='nested').mean('time')

#importing panan0025
gl =glob.glob(src_p0025)
gl=sorted(gl)
print("%i netcdf data files"%len(gl))
p0025_HT=xr.open_mfdataset(gl, concat_dim='time', combine='nested').mean('time')

36 netcdf data files
36 netcdf data files
12 netcdf data files


Interpolating panan01 velocities into the contour, and then the Eddy Kinectic Energy (EKE)

In [9]:
# Panan01
U_along_contour_005 = U_xq.interp(xq=p005_HTmean.lon_along_contour,\
                                 yh=p005_HTmean.lat_along_contour,method='nearest')\
.drop_vars({'xh','xq','yh'})
V_along_contour_005 = V_yq.interp(xh=p005_HTmean.lon_along_contour,\
                                 yq=p005_HTmean.lat_along_contour,method='nearest')\
.drop_vars({'xh','yq','yh'})

In [10]:
EKE_005 = 0.5 * ((U_along_contour_005**2) + (V_along_contour_005**2))
EKE_005.name = 'EKE_m2s2'

In [11]:
#saving file
dir_file = '/g/data/ik11/users/wf4500/Project_panan/GH/Panan_HT_ASC/Processed_data/panan005_rerun/EKE_along_contour/'
name_file = dir_file + 'EKE_along_contour_' + start_time + '.nc'
EKE_005.to_netcdf(name_file)


#visual check
EKE_005

<xarray.DataArray 'EKE_m2s2' (time: 1, rho2_l: 99, contour_index: 12044)> Size: 10MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])
Coordinates:
  * rho2_l         (rho2_l) float64 792B 1.016e+03 1.033e+03 ... 1.038e+03
  * time           (time) object 8B 2001-12-16 12:00:00
  * contour_index  (contour_index) int64 96kB 1 2 3 4 ... 12042 12043 12044